In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [2]:
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.paths import DATA_DIR
from src.parser.parsers import AutoParser

In [3]:
def remove_pua_chars(s: str) -> str:
    return ''.join(c for c in s if not (0xE000 <= ord(c) <= 0xF8FF))

allowed_latex = ["π"] # give context
def remove_latex(text: str):
    text = remove_pua_chars(text)
    # Simply remove every piece of text which 
    tokens = text.split(" ")
    keep = [t for t in tokens if t in allowed_latex or (len(t) > 1 and "uf" not in t)]
    return " ".join(keep)

def process_text(text: str):
    txt = remove_latex(text)
    txt = re.sub(r"(?i)20\d\d .+ exam(ination)?", "", txt) # remove headers
    txt = re.sub(r"(\(?\d+\))? ?marks?", "", txt) # remove leftover marks indicators
    txt = re.sub(r"\(\d+\)*", "", txt) # page numbers and stuff scattered about, remove htem
    txt = txt.replace("  ", " ") # replace double spaces with just one space. 
    return txt

In [12]:
subject = "specialist_mathematics"
exam_path = Path(DATA_DIR / subject / "past_exams" / "2019_2.pdf" )
sd_path = Path(DATA_DIR / subject / "study_design" / f"{subject}_sd.docx")

exam_parser = AutoParser(exam_path)
sd_parser = AutoParser(sd_path)

ex_root = exam_parser.parse()
sd_root = sd_parser.parse()

Exam Preprocessings

In [5]:
qn_level = ex_root.find_node_level()

ex_root.preprocess_text(process_text)
ex_root.collapse(level=qn_level)
ex_root.print_tree()

 [0]: root

Text: 
 [1]: SECTION A – Multiple-choice questionsQuestion 1

Text: Answer questions in pencil on the answer sheet provided for multiple-choice questions. Choose the response that is for the question. correct answer scores 1; an incorrect answer scores 0. Marks will be deducted for incorrect answers. No will be given if more than one answer is completed for any question. Unless otherwise indicated, the diagrams in this book are drawn to scale. Take the to have magnitude ms –2 where 9.8 The graph of does have horizontal asymptote. vertical asymptote. local minimum. vertical axis intercept. point of inflection.
 [1]: SECTION A – Multiple-choice questionsQuestion 2

Text: Answer questions in pencil on the answer sheet provided for multiple-choice questions. Choose the response that is for the question. correct answer scores 1; an incorrect answer scores 0. Marks will be deducted for incorrect answers. No will be given if more than one answer is completed for any question. Unle

In [6]:
qns = ex_root.get_nodes_at_level(qn_level)

SD Preprocessings

In [7]:
sd_root.print_tree()

 [0]: root

Text: 
         [3]: Amendments to study design history

Text:  Authorised and published by the Victorian Curriculum and Assessment Authority
Level 7, 2 Lonsdale Street
Melbourne VIC 3000 ISBN: 978-1-925264-18-0 © Victorian Curriculum and Assessment Authority 2022 No part of this publication may be reproduced except as specified under the Copyright Act 1968 or by permission from the VCAA. Excepting third-party elements, schools may use this resource in accordance with the VCAA educational allowance. For more information read the VCAA copyright policy. The VCAA provides the only official, up-to-date versions of VCAA publications. Details of updates can be found on the VCAA website. This publication may contain copyright material belonging to a third party. Every effort has been made to contact all copyright owners. If you believe that material in this publication is an infringement of your copyright, please email the Copyright Officer. Copyright in materials appearing at any

In [8]:
def std_str(txt: str):
    """ Standardises capitalisation form snake_case to "Name Case". """
    s = txt.replace("_", " ")
    return " ".join([word[0].upper() + word[1:] for word in s.split()])

In [9]:
std_str(subject)
f"Units 3 and 4: {std_str(subject)}"

'Units 3 and 4: Specialist Mathematics'

In [13]:
sd_root.label_search(f"Units 3 and 4: {std_str(subject)}")

In [10]:
for idx, child in enumerate(sd_root.children):
    t = child.label
    print(idx, t, bool(re.search(rf"Units 3 and 4: {std_str(subject)}", t)))

0 Amendments to study design history False
1 Important information False
2 Introduction False
3 Assessment and reporting False
4 Unit 1: Foundation Mathematics False
5 Unit 2: Foundation Mathematics False
6 Unit 1: General Mathematics False
7 Unit 2: General Mathematics False
8 Unit 1: Mathematical Methods False
9 Unit 2: Mathematical Methods False
10 Unit 1: Specialist Mathematics False
11 Unit 2: Specialist Mathematics False
12 Units 3 and 4: Foundation Mathematics False
13 Units 3 and 4: General Mathematics False
14 Units 3 and 4: Mathematical Methods False
15 Units 3 and 4: Specialist Mathematics True


In [ ]:
# First, find subject
if "math" in subject:
    sd_root = sd_root.label_search(rf"Units 3 and 4: {std_str(subject)}")
sd_root.filter_tree(r"Area of Study \d")


AttributeError: 'NoneType' object has no attribute 'filter_tree'